# 04.3 模型保存与实验记录（Checkpoint and Logging）

这一节的目标是让训练结果变得“可恢复  

重点概念

- 检查点（checkpoint）
- 实验配置（experiment config）
- 指标日志（metric logging）
- 结果复现（reproducibility）

## 学习目标

学完后你应该能

1. 理解为什么只保存模型还不够
2. 保存 `model_state
3. 记录训练过程中的 loss 和 accuracy
4. 加载 checkpoint 恢复模型
5. random seed 在复现中的作用

In [ ]:
import csv
import json
import random
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)

## 1. 准备一个最小实验

这里还是用一个小型二分类 toy task，重点不在任务本身，而在记录和恢复流程。  


In [ ]:
config = {
    "seed": 42,
    "batch_size": 32,
    "hidden_dim": 16,
    "lr": 0.1,
    "epochs": 8,
}

set_seed(config["seed"])
workdir = Path(tempfile.mkdtemp(prefix="phase4_checkpoint_logging_"))
best_ckpt_path = workdir / "best_checkpoint.pt"
last_ckpt_path = workdir / "last_checkpoint.pt"
history_json_path = workdir / "history.json"
history_csv_path = workdir / "history.csv"
config_json_path = workdir / "config.json"

x = torch.randn(320, 2)
y = (x[:, 0] - 0.5 * x[:, 1] > 0).long()

train_ds = TensorDataset(x[:256], y[:256])
val_ds = TensorDataset(x[256:], y[256:])
train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False)

print("workdir =", workdir)
print("train size =", len(train_ds))
print("val size =", len(val_ds))

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=16, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_correct = 0
    total_items = 0

    for xb, yb in loader:
        with torch.set_grad_enabled(is_train):
            logits = model(xb)
            loss = loss_fn(logits, yb)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * xb.size(0)
        total_correct += (preds == yb).sum().item()
        total_items += xb.size(0)

    return total_loss / total_items, total_correct / total_items

## 2. 训练并记录 history

一个最低限度但很实用的日志内容通常包括

- epoch
- train loss
- train accuracy
- val loss
- val accuracy

In [ ]:
model = TinyClassifier(hidden_dim=config["hidden_dim"])
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=config["lr"])

history = []
best_val_loss = float("inf")
best_epoch = None

for epoch in range(1, config["epochs"] + 1):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)

    row = {
        "epoch": epoch,
        "train_loss": round(train_loss, 6),
        "train_acc": round(train_acc, 6),
        "val_loss": round(val_loss, 6),
        "val_acc": round(val_acc, 6),
    }
    history.append(row)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "config": config,
                "history": history,
                "best_val_loss": best_val_loss,
            },
            best_ckpt_path,
        )

    print(
        f"epoch={epoch:02d} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

torch.save(
    {
        "epoch": config["epochs"],
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "config": config,
        "history": history,
    },
    last_ckpt_path,
)

print("best_epoch =", best_epoch)
print("best_ckpt_path =", best_ckpt_path)
print("last_ckpt_path =", last_ckpt_path)

## 3. 把 config 和 history 存成文件

为什么要单独存这些文本信息

- 方便快速查看（easy to inspect quickly）
- 不用先加载 `torch` checkpoint（no need to load a `torch` checkpoint first）
- 更方便后续画图和写报告

In [ ]:
config_json_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
history_json_path.write_text(json.dumps(history, indent=2), encoding="utf-8")

with history_csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=history[0].keys())
    writer.writeheader()
    writer.writerows(history)

print("saved files:")
for path in [config_json_path, history_json_path, history_csv_path, best_ckpt_path, last_ckpt_path]:
    print("-", path.name, "exists=", path.exists())

## 4. 加载 checkpoint 恢复模型

这里我们用 `last checkpoint` 来验证恢复是否成功。  


In [ ]:
loaded = torch.load(last_ckpt_path, map_location="cpu")
restored_model = TinyClassifier(hidden_dim=loaded["config"]["hidden_dim"])
restored_model.load_state_dict(loaded["model_state"])
restored_model.eval()
model.eval()

sample_x = x[256:261]
original_logits = model(sample_x)
restored_logits = restored_model(sample_x)
original_preds = original_logits.argmax(dim=1)
restored_preds = restored_logits.argmax(dim=1)

print("loaded epoch =", loaded["epoch"])
print("loaded config =", loaded["config"])
print("original_preds =", original_preds)
print("restored_preds =", restored_preds)
print("predictions identical / 预测是否一致 =", torch.equal(original_preds, restored_preds))

In [ ]:
best_loaded = torch.load(best_ckpt_path, map_location="cpu")
print("best checkpoint epoch =", best_loaded["epoch"])
print("best checkpoint val loss =", best_loaded["best_val_loss"])
print("history length inside checkpoint =", len(best_loaded["history"]))

## 5. 一个实用习惯

最低限度建议你每个实验都留下这几样东西

- `history.json` 或 `history.csv`
- `best_checkpoint.pt`
- 简短实验结论（a short experiment conclusion）

In [ ]:
# 练习 1
# 为什么很多时候只保存 model.state_dict() 还不够？
# Why is saving only model.state_dict() often not enough?

练习 1 参考答案

optimizer state、epoch、实验配置  

这些信息对继续训练和复现实验都很重要。  


In [ ]:
# 练习 2
# `best checkpoint` 和 `last checkpoint` 的区别是什么？
# What is the difference between a `best checkpoint` and a `last checkpoint`?

练习 2 参考答案

- `best checkpoint`：验证集表现最好的那一次
- `last checkpoint`：训练结束时最后一次保存

很多项目两个都会保存，因为它们用途不同。  


## 6. 小结

这一节最重要的收获

1. checkpoint 不只是模型参数，还应该包含恢复训练所需的信息
2. `config` 和 `history` 要尽量单独保存
3. `best checkpoint` 和 `last checkpoint` 通常都值得保留
4. 没有日志和配置，很多实验实际上是不可复现的